# Notebook 6: Run ANPR App on Google Colab
**ANPR System – MIPA UGM Parking Lot**

This notebook runs the full Flask-based ANPR system on Colab and exposes it
via a public ngrok URL. Anyone with the URL can upload a photo and get a result.

**Steps:**
1. Install system dependencies (Tesseract)
2. Install Python packages
3. Upload your project zip
4. Initialize the database
5. Start Flask + ngrok → get public URL

**Before running:** make sure you've zipped the project locally:
```bash
cd ~/
zip -r anpr_mipa.zip anpr-mipa-ugm/ \
  --exclude 'anpr-mipa-ugm/venv/*' \
  --exclude 'anpr-mipa-ugm/data/_extracted/*' \
  --exclude 'anpr-mipa-ugm/data/processed/public_v1/*' \
  --exclude 'anpr-mipa-ugm/__pycache__/*' \
  --exclude 'anpr-mipa-ugm/runs/*' \
  --exclude 'anpr-mipa-ugm/uploads/*' \
  --exclude 'anpr-mipa-ugm/results/*'
```

## Cell 1 — Install Tesseract (system package)

In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-eng 2>/dev/null | tail -3
!tesseract --version

## Cell 2 — Install Python packages

In [ ]:
!pip install -q \
    flask \
    python-dotenv \
    pyyaml \
    ultralytics \
    opencv-python-headless \
    pillow \
    easyocr \
    pytesseract \
    python-Levenshtein \
    scikit-image \
    scipy \
    pyngrok
print('Done.')

## Cell 3 — Upload project zip

A file picker will appear. Select `anpr_mipa.zip` from your Mac.

In [ ]:
from google.colab import files
import zipfile, os

print('Select anpr_mipa.zip ...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Extracting {zip_name} ...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

os.chdir('/content/anpr-mipa-ugm')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

## Cell 4 — Fix config for Colab paths + initialize database

In [ ]:
import os
os.chdir('/content/anpr-mipa-ugm')

# Set env vars so Flask runs correctly in Colab
os.environ['FLASK_DEBUG'] = 'False'
os.environ['FLASK_PORT'] = '5001'
os.environ['FLASK_SECRET_KEY'] = 'colab-demo-secret'

# Initialize the database
!python init_db.py
print('DB ready.')

## Cell 5 — Set your ngrok auth token

1. Sign up free at https://ngrok.com
2. Go to https://dashboard.ngrok.com/get-started/your-authtoken
3. Copy your token and paste it below

In [ ]:
NGROK_TOKEN = 'PASTE_YOUR_TOKEN_HERE'  # <-- replace this

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('ngrok token set.')

## Cell 6 — Start Flask + ngrok

After running this cell, a public URL will appear (e.g. `https://abc123.ngrok.io`).
Open that URL in any browser — that's your ANPR system running live.

In [ ]:
import threading, os
os.chdir('/content/anpr-mipa-ugm')

from pyngrok import ngrok

# Open tunnel BEFORE starting Flask
public_url = ngrok.connect(5001)
print('=' * 60)
print(f'  PUBLIC URL: {public_url}')
print('  Share this URL to access the ANPR system from anywhere.')
print('=' * 60)

# Start Flask in a background thread
def run_flask():
    os.environ['FLASK_DEBUG'] = 'False'
    import app as flask_app
    flask_app.app.run(host='0.0.0.0', port=5001, debug=False, use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()

print('Flask started. Use the URL above.')

## Cell 7 — Keep alive (run this to prevent Colab from timing out)

Colab disconnects after ~90 minutes of inactivity. Run this cell to keep it alive
while you're doing a demo. Stop it (square button) when done.

In [ ]:
import time
print('Keeping alive... (stop this cell to shut down)')
while True:
    time.sleep(30)
    print('.', end='', flush=True)